# Vígil.ia — Validação por classe (multi-modelo, sem treino)

Compara **vários detectores** (YOLO11 s/l/x e RT-DETR) em vídeos onde a classe é
conhecida pela **pasta**: cada subpasta tem vídeo(s) só daquela classe (grãos
multi-grão, espalhados, sem se encostar).

Cada modelo roda em cada vídeo, rastreia cada grão e fecha a classe por **voto
temporal** (peso = confiança). Como a verdade é a pasta, dá pra montar a **matriz
de confusão** por modelo e ranquear — sem anotar, sem treinar, sem salvar.

> Só validação. Não altera nem salva nenhum modelo.
> Detectores (multi-grão): `soja_yolo11s_det_v3.pt`, `soja_yolo11l_v3.pt`,
> `soja_yolo11x_v3.pt`, `soja_rtdetr_ft_v3.pt`, …
> ⚠️ Os `soja_yolo11s_finetuned.pt` / `_best.pt` são **classificadores** (1 grão) —
> NÃO entram aqui.
>
> Estrutura no Drive (só as pastas com vídeo entram):
> ```
> <VAL_ROOT>/Intacto/ Quebrado/ Imaturo/ Manchado/ Casca danificada/
> ```

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU! (Ambiente de execução -> Alterar tipo -> GPU)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, unicodedata
from google.colab import drive
drive.mount('/content/drive')

MODELS_DIR = '/content/drive/MyDrive'

# --- detectores a comparar (só entram os que EXISTIREM no Drive) ---
# (tag amigável, nome do arquivo .pt). RT-DETR é detectado pelo nome ('rtdetr').
MODEL_CANDS = [
    ('yolo11x v3 (campeão)', 'soja_yolo11x_v3.pt'),
    ('yolo11l v3',           'soja_yolo11l_v3.pt'),
    ('yolo11s v3',           'soja_yolo11s_det_v3.pt'),
    ('rtdetr ft v3',         'soja_rtdetr_ft_v3.pt'),
    ('rtdetr ft v3 disc',    'soja_rtdetr_ft_v3_disc.pt'),
    ('yolo11x box10',        'soja_yolo11x_v3_box10.pt'),
    ('yolo11x active',       'soja_yolo11x_active.pt'),
]

# --- raiz com UMA subpasta por classe (cada uma com vídeo(s) só daquela classe) ---
# atenção ao acento em "Vídeos". Troque p/ .../Validação quando ela tiver vídeos.
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
assert os.path.isdir(VAL_ROOT), (
    f'pasta não encontrada: {VAL_ROOT}\n'
    'confira com:  !ls -R "/content/drive/MyDrive/Vídeos para treino"')

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
PT = {'broken': 'quebrado', 'immature': 'imaturo', 'intact': 'intacto',
      'skin-damaged': 'casca-dan', 'spotted': 'manchado'}

# casa o nome da pasta (PT/EN, com ou sem acento) -> índice da classe
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'],
           4: ['spotted', 'manchad']}

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(name):
    n = norm(name)
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
CONF, IMGSZ, IOU = 0.35, 640, 0.5   # mesmos parâmetros da inspeção
print('config ok | VAL_ROOT =', VAL_ROOT)

## 1. Descobrir os vídeos de cada classe

In [ ]:
# varre VAL_ROOT: cada subpasta que casa com uma classe e tem vídeo entra na validação
by_class = {}   # nome da classe -> [caminhos de vídeo]
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    c = class_of(entry)
    if c is None:
        print(f'  (ignorada)  "{entry}" não casa com nenhuma das 5 classes')
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if not vids:
        print(f'  (sem vídeo) "{entry}" -> {NAMES[c]}')
        continue
    by_class[NAMES[c]] = vids
    print(f'  "{entry}" -> {NAMES[c]:12s}: {len(vids)} vídeo(s)')

assert by_class, 'Nenhuma pasta de classe com vídeo em VAL_ROOT — confira a estrutura.'
print('\nclasses que serão validadas:', list(by_class))

## 2. Validar — cada modelo roda e vota a classe de cada grão

Carrega YOLO com `YOLO()` e RT-DETR com `RTDETR()` (+ patch de NMS class-agnostic,
igual aos seus outros notebooks — sem ele o RT-DETR gera caixas sobrepostas).
Mede a **acurácia de classe dos grãos que o modelo detecta** (grão não detectado
não aparece — é acurácia de classe, não recall).

In [ ]:
from collections import defaultdict, Counter
from ultralytics import YOLO, RTDETR

_rtdetr_patched = False
def _patch_rtdetr_nms(iou_nms=0.6):
    """RT-DETR não faz NMS por padrão -> aplica NMS class-agnostic (evita caixa dupla)."""
    global _rtdetr_patched
    if _rtdetr_patched:
        return
    import torchvision
    from ultralytics.models.rtdetr.predict import RTDETRPredictor
    if not hasattr(RTDETRPredictor, '_pp_orig'):
        RTDETRPredictor._pp_orig = RTDETRPredictor.postprocess

        def _pp_nms(self, preds, img, orig_imgs):
            results = RTDETRPredictor._pp_orig(self, preds, img, orig_imgs)
            for r in results:
                if len(r.boxes) > 1:
                    keep = torchvision.ops.nms(r.boxes.xyxy, r.boxes.conf, iou_nms)
                    r.update(boxes=r.boxes.data[keep])
            return results

        RTDETRPredictor.postprocess = _pp_nms
    _rtdetr_patched = True

def load_model(path):
    if 'rtdetr' in os.path.basename(path).lower():
        _patch_rtdetr_nms()
        return RTDETR(path)
    return YOLO(path)

def validar(path):
    model = load_model(path)
    y_true, y_pred = [], []
    for true_cls, vids in by_class.items():
        votes = defaultdict(Counter)      # (vídeo, id do grão) -> votos por classe
        for vid in vids:
            for r in model.track(source=vid, imgsz=IMGSZ, iou=IOU, conf=CONF,
                                 agnostic_nms=True, tracker='bytetrack.yaml',
                                 stream=True, verbose=False):
                if r.boxes.id is None:
                    continue
                for tid, c, cf in zip(r.boxes.id.int().tolist(),
                                      r.boxes.cls.int().tolist(),
                                      r.boxes.conf.tolist()):
                    votes[(vid, tid)][NAMES[c]] += cf
        for cnt in votes.values():
            y_true.append(true_cls)
            y_pred.append(cnt.most_common(1)[0][0])
    return y_true, y_pred

# só os modelos que existem no Drive
MODELS = [(tag, os.path.join(MODELS_DIR, fn)) for tag, fn in MODEL_CANDS
          if os.path.exists(os.path.join(MODELS_DIR, fn))]
faltando = [fn for _, fn in MODEL_CANDS if not os.path.exists(os.path.join(MODELS_DIR, fn))]
assert MODELS, 'Nenhum modelo da lista existe no Drive — confira MODEL_CANDS/MODELS_DIR.'
print('serão validados:', [t for t, _ in MODELS])
if faltando:
    print('não achei no Drive (pulados):', faltando)

results = {}
for tag, path in MODELS:
    print(f'\n>>> {tag}')
    yt, yp = validar(path)
    results[tag] = (yt, yp)
    acc = 100 * sum(t == p for t, p in zip(yt, yp)) / max(len(yt), 1)
    print(f'    {len(yt)} grãos | acurácia {acc:.1f}%')
print('\nfeito.')

## 3. Resultados — ranking + acurácia por classe + matriz de confusão

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def confusao(y_true, y_pred):
    idx = {c: i for i, c in enumerate(NAMES)}
    M = np.zeros((5, 5), int)
    for t, p in zip(y_true, y_pred):
        M[idx[t], idx[p]] += 1
    return M

def acc_de(res):
    yt, yp = res
    return 100 * sum(t == p for t, p in zip(yt, yp)) / max(len(yt), 1)

# ---- ranking geral ----
print('===== RANKING (acurácia geral) =====')
for tag, res in sorted(results.items(), key=lambda kv: -acc_de(kv[1])):
    yt = res[0]
    print(f'  {tag:24s} {acc_de(res):5.1f}%   ({len(yt)} grãos)')
print()

# ---- detalhe + matriz por modelo ----
for tag, (y_true, y_pred) in results.items():
    M = confusao(y_true, y_pred)
    n = len(y_true)
    print(f'\n================  {tag}  ({acc_de((y_true, y_pred)):.1f}%)  ================')
    for i, c in enumerate(NAMES):
        tot = M[i].sum()
        if tot == 0:
            continue
        certos = M[i, i]
        errs = {NAMES[j]: int(M[i, j]) for j in range(5) if j != i and M[i, j] > 0}
        print(f'  {c:12s}: {certos}/{tot} ({100*certos/tot:.1f}%)   confundiu -> {errs if errs else "—"}')

    fig, ax = plt.subplots(figsize=(5.6, 4.9))
    ax.imshow(M, cmap='Blues')
    ax.set_xticks(range(5)); ax.set_xticklabels([PT[c] for c in NAMES], rotation=45, ha='right')
    ax.set_yticks(range(5)); ax.set_yticklabels([PT[c] for c in NAMES])
    ax.set_xlabel('previsto pelo modelo'); ax.set_ylabel('verdadeiro (pasta)')
    ax.set_title(f'{tag} — matriz de confusão')
    thr = M.max() / 2 if M.max() else 1
    for i in range(5):
        for j in range(5):
            if M[i, j]:
                ax.text(j, i, int(M[i, j]), ha='center', va='center',
                        color='white' if M[i, j] > thr else 'black')
    plt.tight_layout(); plt.show()

print('\n⚠️ Se só a pasta Intacto tiver vídeo, esse ranking mede só "quem menos')
print('   flagra intacto como defeito". Um modelo que dissesse SEMPRE intacto ganharia')
print('   aqui e seria inútil. Suba as outras 4 classes p/ o ranking valer de verdade.')